<a target="_blank" href="https://colab.research.google.com/github/rajshah4/LLM-Evaluation/blob/main/ragas_quickstart.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Welcome to the ragas quickstart. We're going to get you up and running with ragas as qickly as you can so that you can go back to improving your Retrieval Augmented Generation pipelines while this library makes sure your changes are improving your entire pipeline.

to kick things of lets start with the data

In [ ]:
# %pip -q install ragas pandas datasets

Ragas also uses OpenAI for running some metrics so make sure you have your openai key ready and available in your environment

In [1]:
import os
import getpass
from dotenv import load_dotenv
import json
import re
import pandas as pd

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
# open_ai_key = getpass.getpass('Enter your OPENAI API Key')
# os.environ['OPENAI_API_KEY'] = open_ai_key

## The Data

Ragas performs a `ground_truth` free evaluation of your RAG pipelines. This is because for most people building a gold labeled dataset which represents in the distribution they get in production is a very expensive process.

Hence to work with ragas all you need are the following data
- question: `list[str]` - These are the questions you RAG pipeline will be evaluated on.
- answer: `list[str]` - The answer generated from the RAG pipeline and give to the user.
- contexts: `list[list[str]]` - The contexts which where passed into the LLM to answer the question.

Ideally your list of questions should reflect the questions your users give, including those that you have been problamatic in the past.

Here we're using an example dataset from on of the baselines we created for the [Financial Opinion Mining and Question Answering (fiqa) Dataset](https://sites.google.com/view/fiqa/) we created. If you want to want to know more about the baseline, feel free to check the `experiements/baseline` section

In [2]:
df = pd.read_csv('/home/dedsec995/Data_Science_Learning/dataset/day33/SAGE_Melodies.csv')

In [9]:
type(df['CONTEXT'][0])

str

In [ ]:
def extract_clean_text(json_input):
    data = json_input
    contexts = data.get("contexts_found", [])
    
    cleaned_blocks = []
    for block in contexts:
        cleaned_text = re.sub(r'^\s*\d+\s*:\s*', '', block)
        cleaned_blocks.append(cleaned_text.strip())
    return "\n\n".join(cleaned_blocks)


if __name__ == "__main__":
    sample_json = {
        "status": "success",
        "contexts_found": {
            "0": "This is the first passage.",
            "1": "This is the second passage.",
            "2": "This is the third passage."
        }
    }

    if isinstance(sample_json["contexts_found"], dict):
        sample_json["contexts_found"] = list(sample_json["contexts_found"].values())

    output = extract_clean_text(sample_json)
    print(output)


In [ ]:
# data
from datasets import load_dataset

fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")
fiqa_eval

README.md: 0.00B [00:00, ?B/s]

data/ragas_eval/baseline.parquet:   0%|          | 0.00/106k [00:00<?, ?B/s]

  2025-10-22T15:08:01.683278Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7f97d171ea70>), traceback: Some(<traceback object at 0x7f9682c0aa00>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28



## Metrics

Ragas measures your pipeline's performance against two dimensions

1. Faithfulness: measures the factual consistency of the generated answer against the given context.
2. Relevancy: measures how relevant retrieved contexts and the generated answer are to the question.

Through repeated experiments, we have found that the quality of a RAG pipeline is highly dependent on these two dimensions. The final `ragas_score` is the harmonic mean of these two factors.

now lets import these metrics and understand more about what they denote

In [ ]:
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
)

here you can see that we are using 4 metrics, but what do the represent?

1. answer_relevancy - a measure of how relevant the answer is to the question

2. faithfulness - the factual consistency of the answer to the context base on the question.

3. context_recall: measures the ability of the retriever to retrieve all the necessary information needed to answer the question.

4. context_precision - a measure of how relevant the retrieved context is to the question. Conveys quality of the retrieval pipeline.

**Note:** *by default these metrics are using OpenAI's API to compute the score. If you using this metric make sure you set the environment key `OPENAI_API_KEY` with your API key. You can also try other LLMs for evaluation, check the [llm guide](./guides/llms.ipynb) to learn more*

If you're interested in learning more, feel free to check the [docs](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)

## Evaluation

Running the evalutation is as simple as calling evaluate on the `Dataset` with the metrics of your choice.

In [ ]:
from ragas import evaluate

result = evaluate(
    fiqa_eval["baseline"].select(range(1)),
    metrics=[
        context_precision,
        faithfulness,
        answer_relevancy,
        context_recall
    ],
)

result

and there you have the it, all the scores you need. `ragas_score` gives you a single metric that you can use while the other onces measure the different parts of your pipeline.

now if we want to dig into the results and figure out examples where your pipeline performed worse or really good you can easily convert it into a pandas array and use your standard analytics tools too!

In [ ]:
df = result.to_pandas()
df.head()

And thats it!

You can check out the [ragas in action] notebook to get a feel of what is like to use it while trying to improve your pipelines.

if you have any suggestion/feedbacks/things your not happy about, please do share it in the [issue section](https://github.com/explodinggradients/ragas/issues). We love hearing from you 😁